In [2]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("../Data/heart.csv")
df.head()

,id,age,sex,dataset,cp,trestbps,chol,fbs,restecg,thalch,exang,oldpeak,slope,ca,thal,num
0,1,63,Male,Cleveland,typical angina,145.0,233.0,True,lv hypertrophy,150.0,False,2.3,downsloping,0.0,fixed defect,0
1,2,67,Male,Cleveland,asymptomatic,160.0,286.0,False,lv hypertrophy,108.0,True,1.5,flat,3.0,normal,2
2,3,67,Male,Cleveland,asymptomatic,120.0,229.0,False,lv hypertrophy,129.0,True,2.6,flat,2.0,reversable defect,1
3,4,37,Male,Cleveland,non-anginal,130.0,250.0,False,normal,187.0,False,3.5,downsloping,0.0,normal,0
4,5,41,Female,Cleveland,atypical angina,130.0,204.0,False,lv hypertrophy,172.0,False,1.4,upsloping,0.0,normal,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 920 entries, 0 to 919
Data columns (total 16 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        920 non-null    int64  
 1   age       920 non-null    int64  
 2   sex       920 non-null    object 
 3   dataset   920 non-null    object 
 4   cp        920 non-null    object 
 5   trestbps  861 non-null    float64
 6   chol      890 non-null    float64
 7   fbs       830 non-null    object 
 8   restecg   918 non-null    object 
 9   thalch    865 non-null    float64
 10  exang     865 non-null    object 
 11  oldpeak   858 non-null    float64
 12  slope     611 non-null    object 
 13  ca        309 non-null    float64
 14  thal      434 non-null    object 
 15  num       920 non-null    int64  
dtypes: float64(5), int64(3), object(8)
memory usage: 115.1+ KB


In [4]:
df["num"].value_counts()

num
0    411
1    265
2    109
3    107
4     28
Name: count, dtype: int64

In [5]:
df["num"] = (df["num"] > 0).astype(int)
df = df.drop(columns="id")

In [6]:
X = df.drop(columns="num")
y = df["num"]

In [7]:
numeric_col = X.select_dtypes(include=["int64", "float64"]).columns
categorical_col = X.select_dtypes(include=["object"]).columns

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

numeric_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="median")),
    ("Scale", StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ("impute", SimpleImputer(strategy="most_frequent")),
    ("encode", OneHotEncoder(handle_unknown="ignore"))
])

In [9]:
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer([
    ("num", numeric_pipeline, numeric_col),
    ("cat", categorical_pipeline, categorical_col)
])

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

model_lr = Pipeline(steps=[
    ("preprocess", preprocessing),
    ("model", LogisticRegression(max_iter=1000))
])

scores = cross_val_score(model_lr, X, y, cv=5)

print("Accuracy:", scores.mean())
print("Std:", scores.std())

Accuracy: 0.7836956521739131
Std: 0.10674333649731231


In [12]:
model_lr.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [13]:
probs = model_lr.predict_proba(X_test)[:, 1]
pred = model_lr.predict(X_test)

In [14]:
from sklearn.metrics import accuracy_score, classification_report

for t in [0.3, 0.4, 0.5, 0.6]:
    pred_t = (probs > t).astype(int)
    print(f"\nThreshold: {t}")
    print(classification_report(y_test, pred_t))
          

# print("Accuracy", accuracy_score(y_test, pred_new))
# print(classification_report(y_test, pred_new))


Threshold: 0.3
              precision    recall  f1-score   support

           0       0.93      0.67      0.78        82
           1       0.78      0.96      0.86       102

    accuracy                           0.83       184
   macro avg       0.86      0.82      0.82       184
weighted avg       0.85      0.83      0.83       184


Threshold: 0.4
              precision    recall  f1-score   support

           0       0.87      0.72      0.79        82
           1       0.80      0.91      0.85       102

    accuracy                           0.83       184
   macro avg       0.83      0.82      0.82       184
weighted avg       0.83      0.83      0.82       184


Threshold: 0.5
              precision    recall  f1-score   support

           0       0.85      0.77      0.81        82
           1       0.83      0.89      0.86       102

    accuracy                           0.84       184
   macro avg       0.84      0.83      0.83       184
weighted avg       0.84   

In [15]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, pred)
print(cm)

[[63 19]
 [11 91]]


In [16]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, probs)
print("ROC-AUC:", auc)

ROC-AUC: 0.9203730272596844


In [17]:
from sklearn.ensemble import RandomForestClassifier

model_rf = Pipeline(steps=[
    ("preprocess", preprocessing),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=5,
        random_state=42
    ))
])

In [18]:
from sklearn.model_selection import cross_val_score

scores = cross_val_score(model_rf, X_train, y_train, cv=5)

print("RF scores:", scores.mean())
print("Std:", scores.std())

RF scores: 0.8192682478396766
Std: 0.018269905423472683


In [19]:
model_rf.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [20]:
pred_rf = model_rf.predict(X_test)
probs_rf = model_rf.predict_proba(X_test)[:,1]

In [21]:
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, pred_rf))

auc_rf = roc_auc_score(y_test, probs_rf)
print("RF ROC-AUC:", auc_rf)

              precision    recall  f1-score   support

           0       0.87      0.80      0.84        82
           1       0.85      0.90      0.88       102

    accuracy                           0.86       184
   macro avg       0.86      0.85      0.86       184
weighted avg       0.86      0.86      0.86       184

RF ROC-AUC: 0.9348397895743663


In [22]:
for t in [0.3, 0.4, 0.5]:
    pred_t = (probs_rf > t).astype(int)

    print(f"\nThreshold {t}")
    print(classification_report(y_test, pred_t))


Threshold 0.3
              precision    recall  f1-score   support

           0       0.96      0.59      0.73        82
           1       0.75      0.98      0.85       102

    accuracy                           0.80       184
   macro avg       0.85      0.78      0.79       184
weighted avg       0.84      0.80      0.79       184


Threshold 0.4
              precision    recall  f1-score   support

           0       0.90      0.70      0.79        82
           1       0.79      0.94      0.86       102

    accuracy                           0.83       184
   macro avg       0.85      0.82      0.82       184
weighted avg       0.84      0.83      0.83       184


Threshold 0.5
              precision    recall  f1-score   support

           0       0.87      0.80      0.84        82
           1       0.85      0.90      0.88       102

    accuracy                           0.86       184
   macro avg       0.86      0.85      0.86       184
weighted avg       0.86      

In [23]:
from xgboost import XGBClassifier

model_xgb = Pipeline(steps=[
    ("preprocess", preprocessing),
    ("model", XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        random_state=42,
    ))
])

In [24]:
scores = cross_val_score(model_xgb, X_train, y_train, cv=5)

print("XGB scores:", scores.mean())
print("XGB std:", scores.std())

XGB scores: 0.811169332597904
XGB std: 0.030039164598863785


In [25]:
model_xgb.fit(X_train, y_train)

,steps,"[('preprocess', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [26]:
pred_xgb = model_xgb.predict(X_test)
probs_xgb = model_xgb.predict_proba(X_test)[:,1]

print(classification_report(y_test, pred_xgb))
print("XGB ROC-AUC:", roc_auc_score(y_test, probs_xgb))

              precision    recall  f1-score   support

           0       0.88      0.80      0.84        82
           1       0.85      0.91      0.88       102

    accuracy                           0.86       184
   macro avg       0.87      0.86      0.86       184
weighted avg       0.87      0.86      0.86       184

XGB ROC-AUC: 0.9097321855571497


In [27]:
for t in [0.3, 0.4, 0.5]:
    pred_t = (probs_xgb > t).astype(int)

    print(f"\nThreshold {t}")
    print(classification_report(y_test, pred_t))


Threshold 0.3
              precision    recall  f1-score   support

           0       0.93      0.63      0.75        82
           1       0.77      0.96      0.85       102

    accuracy                           0.82       184
   macro avg       0.85      0.80      0.80       184
weighted avg       0.84      0.82      0.81       184


Threshold 0.4
              precision    recall  f1-score   support

           0       0.90      0.78      0.84        82
           1       0.84      0.93      0.88       102

    accuracy                           0.86       184
   macro avg       0.87      0.86      0.86       184
weighted avg       0.87      0.86      0.86       184


Threshold 0.5
              precision    recall  f1-score   support

           0       0.88      0.80      0.84        82
           1       0.85      0.91      0.88       102

    accuracy                           0.86       184
   macro avg       0.87      0.86      0.86       184
weighted avg       0.87      

In [28]:
importances = model_rf.named_steps["model"].feature_importances_
features = model_rf.named_steps["preprocess"].get_feature_names_out()

feat_imp = pd.DataFrame({
    "feature": features,
    "importance": importances
}).sort_values(by="importance", ascending=False)

print(feat_imp.head(10))

                     feature  importance
12      cat__cp_asymptomatic    0.157455
22           cat__exang_True    0.107704
4               num__oldpeak    0.083388
21          cat__exang_False    0.080988
3                num__thalch    0.072867
2                  num__chol    0.071158
0                   num__age    0.064554
13   cat__cp_atypical angina    0.060550
10  cat__dataset_Switzerland    0.041563
6            cat__sex_Female    0.033277
